### MNIST Image Classifier using CNN
- Dataset used is MNIST Dataset downloaded from [Kaggle](https://www.kaggle.com/datasets/hojjatk/mnist-dataset)
- The classifier is based on a CNN deep learning architecture developed using pytorch

#### Imports

In [11]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np # For numerical operations
import struct # To read the binary file format of the dataset
# from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd


#### Specify the configuration

In [2]:
import torch

WORKING_FOLDER_PATH = "."

class Config:
    line_divider = "=" * 50
    next_line = "\n"
    random_state = 100
    
    img_width = 28
    img_height = 28
    first_layer_conv_width = 3
    first_layer_conv_height = 3
    dense_layer_size = 100
    epochs = 10
    batch_size = 64
    learning_rate = 0.001
    
    batch_size = 32
    num_filters = 16
    kernel_size = 3
    hidden_dim = 250
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_fraction_split = 0.98
    
    # Data paths
    train_images_path = f"{WORKING_FOLDER_PATH}/data/minst-image-dataset/train-images-idx3-ubyte/train-images-idx3-ubyte"
    train_labels_path = f"{WORKING_FOLDER_PATH}/data/minst-image-dataset/train-labels-idx1-ubyte/train-labels-idx1-ubyte"
    test_images_path = f"{WORKING_FOLDER_PATH}/data/minst-image-dataset/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte"
    test_labels_path = f"{WORKING_FOLDER_PATH}/data/minst-image-dataset/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte"

    # Model path
    model_cnn_path = f"{WORKING_FOLDER_PATH}/models/mnist_simple_cnn.pth"
    

    is_use_pretrained_model = True

config = Config()

#### Data provider logic

In [3]:
def read_idx(filename):
    """
    Reads the binary IDX file format used by MNIST.
    """
    with open(filename, 'rb') as f:
        # Read the header to understand file structure (magic number, dimensions, etc.)
        zero, data_type, dims = struct.unpack('>HBB', f.read(4))
        shape = tuple(struct.unpack('>I', f.read(4))[0] for d in range(dims))
        # Read the rest of the data and reshape it
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(shape)

def get_datasets() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    try:
        # Load the data
        print(f"x_train path: {config.train_images_path}")
        print(f"x_test path: {config.test_images_path}")
        print(f"y_train path: {config.train_labels_path}")
        print(f"y_test path: {config.test_labels_path}")

        x_train_ = read_idx(config.train_images_path)
        y_train_ = read_idx(config.train_labels_path)
        x_test = read_idx(config.test_images_path)
        y_test = read_idx(config.test_labels_path)

        n_train_samples = int(x_train_.shape[0] * config.train_fraction_split)
        print(f"Splitting training data into {n_train_samples} samples for training and {x_train_.shape[0] - n_train_samples} samples for unused set.")
        x_train, y_train = x_train_[:n_train_samples], y_train_[:n_train_samples]
        x_unused, y_unused = x_train_[n_train_samples:], y_train_[n_train_samples:]
        
        print("Data loaded successfully!")
        print(f"Training Data Shape: {x_train.shape}")
        print(f"Training Labels Shape: {y_train.shape}")
        print(f"Test Data Shape: {x_test.shape}")
        print(f"Test Labels Shape: {y_test.shape}")
        print(f"Unused Data Shape: {x_unused.shape}")
        print(f"Unused Labels Shape: {y_unused.shape}")
         
    except FileNotFoundError:
        print("Error: Files not found. Please check if the dataset is added to the notebook correctly.")
        return None, None, None, None, None, None   
    
    return x_train, y_train, x_test, y_test, x_unused, y_unused

x_train, y_train, x_test, y_test, x_unused, y_unused = get_datasets()

x_train path: ./data/minst-image-dataset/train-images-idx3-ubyte/train-images-idx3-ubyte
x_test path: ./data/minst-image-dataset/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte
y_train path: ./data/minst-image-dataset/train-labels-idx1-ubyte/train-labels-idx1-ubyte
y_test path: ./data/minst-image-dataset/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte
Splitting training data into 58800 samples for training and 1200 samples for unused set.
Data loaded successfully!
Training Data Shape: (58800, 28, 28)
Training Labels Shape: (58800,)
Test Data Shape: (10000, 28, 28)
Test Labels Shape: (10000,)
Unused Data Shape: (1200, 28, 28)
Unused Labels Shape: (1200,)


In [4]:
x_unused[:2]

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(2, 28, 28), dtype=uint8)

#### Create Dataloader for the trai/test datasets

In [5]:
train_dataset = torch.utils.data.TensorDataset(torch.tensor(x_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset = torch.utils.data.TensorDataset(torch.tensor(x_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
unused_dataset = torch.utils.data.TensorDataset(torch.tensor(x_unused, dtype=torch.float32), torch.tensor(y_unused, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
unused_loader = DataLoader(unused_dataset, batch_size=config.batch_size, shuffle=False)


#### Specify the CNN network for the classifier model

In [6]:
class MnistNet(nn.Module):
    def __init__(self, config):
        super(MnistNet, self).__init__()
        # PyTorch Conv2d: (in_channels, out_channels, kernel_size)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(config.first_layer_conv_width, config.first_layer_conv_height))
        self.pool = nn.MaxPool2d(2, 2)
        
        # Calculate flatten size: (28-3+1)/2 = 13 -> 32 * 13 * 13 = 5408
        self.fc1 = nn.Linear(32 * 13 * 13, config.dense_layer_size)
        self.fc2 = nn.Linear(config.dense_layer_size, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        # Note: CrossEntropyLoss in PyTorch expects raw logits, not Softmax
        x = self.fc2(x)
        return x


#### Create the CNN Classifier

In [9]:
class MnistCNNClassifier:
    def __init__(self, config):
        self.config = config
        self.model = MnistNet(config).to(config.device)
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=config.learning_rate)
    
    def train(self, train_loader):
        self.model.train()
        for epoch in range(self.config.epochs):
            running_loss = 0.0
            for i, (inputs, labels) in enumerate(train_loader):
                inputs, labels = inputs.to(self.config.device), labels.to(self.config.device)
                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                running_loss += loss.item()
            print(f"Epoch {epoch+1}/{self.config.epochs}, Loss: {running_loss/len(train_loader):.4f}")

    def evaluate(self, test_loader):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(self.config.device), labels.to(self.config.device)
                outputs = self.model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = correct / total
        print(f"Test Accuracy: {accuracy:.4f}")

    def save_model(self, path):
        torch.save(self.model.state_dict(), path)

    def load_model(self, path):
        self.model.load_state_dict(torch.load(path, map_location=self.config.device))
        self.model.to(self.config.device)

    def predict(self, unused_loader) -> list[int]:
        predicted_labels = []
        self.model.eval()
        with torch.no_grad():
            for inputs, labels in unused_loader:
                inputs = inputs.to(self.config.device)
                outputs = self.model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                predicted_labels.append(predicted.item())
        return predicted_labels

#### Run the classier:
- run the train-validate cycle
- use the trained classifier to predict the class of unseen images 

In [12]:
# Example usage:

classifier = MnistCNNClassifier(config)
if config.is_use_pretrained_model and os.path.exists(config.model_cnn_path):
    print("Loading pretrained model...")
    classifier.load_model(config.model_cnn_path)
else:
    print("Training model from scratch...")
    classifier.train(train_loader)
    classifier.save_model(config.model_cnn_path)
classifier.evaluate(test_loader)
predicted_labels = classifier.predict(unused_loader)
print(f"Predicted labels for unused set: {predicted_labels}")


Training model from scratch...


RuntimeError: Given groups=1, weight of size [32, 1, 3, 3], expected input[1, 32, 28, 28] to have 1 channels, but got 32 channels instead